# Introdução a Openai API

In [ ]:
!pip install openai

In [ ]:
import openai
from openai import OpenAI
from google.colab import userdata


In [ ]:
# uso simples
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

response = client.responses.create(
    model="gpt-4o-mini",
    input="o que é a vida?",
    temperature=0,
    max_output_tokens=500,

)

print(response.output_text)

A vida é um conceito complexo que abrange uma série de características e fenômenos. Em termos biológicos, a vida é geralmente definida por características como crescimento, reprodução, resposta a estímulos, metabolismo e adaptação ao ambiente. 

Filosoficamente, a vida pode ser vista como uma busca por significado, experiências e relações. Cada cultura e indivíduo pode ter sua própria interpretação do que é viver plenamente, envolvendo aspectos como felicidade, amor, aprendizado e contribuição para a sociedade.

Em resumo, a vida é tanto um fenômeno biológico quanto uma experiência subjetiva, rica em significados e interpretações.


In [ ]:
# uso simples
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

response = client.responses.create(
    model="gpt-4o-mini",
    input="Quantos Os tem a palavra capivara",
    temperature=0,
    max_output_tokens=100

)

print(response.output_text)

A palavra "capivara" tem duas letras "a" e nenhuma letra "o". Portanto, a resposta é que não há "os" na palavra "capivara".


In [ ]:
#Usando outros Roles

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "responda como um mineiro"
        },
        {
            "role": "user",
            "content": "Ponto e vírgula são opcionais em JavaScript?"
        }
    ]
)

print(completion.choices[0].message.content)

Uai, sô! No JavaScript, o ponto e vírgula é meio que opcional, sim. O JavaScript tem uma parada chamada "Automatic Semicolon Insertion" que tenta adicionar os pontos e vírgulas automaticamente em certos lugares. Mas, olha, é bom usar o ponto e vírgula em fim de linha pra evitar lá as confusões, viu? Não custa nada fazer isso pra deixar o código mais bonitinho e evitar erro de entendimento. No final das contas, é mais questão de estilo e faça como preferir, mas é sempre bom se preocupar com a clareza do código, né?


## Exercício, altere o role system e veja a alteração de comportamento das respostas

# Zero-shot

In [ ]:
from openai import OpenAI
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

client = OpenAI()


completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Você é um especialista em classificação de sentimentos que sempre classifica um texto em um dos seguintes valores: positivo, negativo ou neutro"
        },
        {
            "role": "user",
            "content": "este restaurante é bom"
        }

    ]
)
print(completion.choices[0].message.content)


Positivo


#few-shot

In [ ]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Você é um especialista em classificação de sentimentos que sempre classifica um texto usando a logica definida abaixo, em um dos seguintes valores: positivo, negativo ou neutro"
        },

        {
            "role": "user",
            "content": "este restaurante é péssimo"
        },

        {
            "role": "assistant",
            "content": "negativo"
        },
        {
            "role": "user",
            "content": "este restaurante é ótimo"
        },

        {
            "role": "assistant",
            "content": "positivo"
        },
        {
            "role": "user",
            "content": "a comida é mais ou menos boa"
        },

        {
            "role": "assistant",
            "content": "neutro"
        },

        {
            "role": "user",
            "content": "este restaurante é ok"
        }

    ]
)
print(completion.choices[0].message.content)


# Exercício:

Altere o código acima para que o LLM responda:

Negativo -> quando for positivo

Positivo -> quando for neutro

Neutro -> quando for negativo

# Saídas estruturadas

In [ ]:
#saidas estruturadas
from pydantic import BaseModel
from openai import OpenAI
from google.colab import userdata
import os
from datetime import date

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

hoje = str(date.today())

client = OpenAI()

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": f"Extract the event information. Extract date in format dd/mm/Y. Consider today as {hoje}" },
        {"role": "user", "content": "Alice e Bob vão ma feira de ciencias na sexta-feira"},
    ],
    response_format=CalendarEvent,
)

event = completion.choices[0].message.parsed
event

CalendarEvent(name='Feira de Ciências', date='01/04/2026', participants=['Alice', 'Bob'])

# Exercício, altere o código acima para extrair os o nome, cpf e o número de telefone da seguinte frase:

Meu nome é João, cpf 111.111.111-99 e o meu numero é 38 5533-3355.

# Exercício, altere o código acima para extrair os o nome, número de telefone, data e hora do agendamento da seguinte frase:

Meu nome é João, meu numero é 38 5533-3355 e eu gostaria de agendar um horário amanhã as 10hrs.

# Trabalhando com modelos multimodais

In [ ]:
#usando imagens no prompt:

from PIL import Image
import requests

url = "https://marketplace.canva.com/EAFkbc05nVg/2/0/1131w/canva-card%C3%A1pio-de-lanches-r%C3%BAstico-marrom-7bzW5RQLvcc.jpg"
im = Image.open(requests.get(url, stream=True).raw)
im

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "What is in the image?"},
            {
                "type": "image_url",
                "image_url": {
                    "url": url,
                },
            },
        ],
    }],
)

print(response.choices[0].message.content)

# 1 - Faça um prompt para extrair todos os pratos, preços e ingredientes de cada item do cardápio

# 2 - Modifique o código acima para receber uma imagem de uma obra de arte e descrevê-la

# 3 - Utilize a saida do modelo para extrair dados estruturados de imagens de obras de arte. Como estilo, palavras chave, cores mais presentes, etc.


# (Re)construindo o chatgpt

In [ ]:
msgs

[{'role': 'system',
  'content': 'Você é o Zé, o assistente virtual.\n            '},
 {'role': 'user', 'content': 'ola'},
 {'role': 'assistant', 'content': 'Olá! Como posso ajudar você hoje?'},
 {'role': 'user', 'content': 'quanto é 2+2?'},
 {'role': 'assistant',
  'content': '2 + 2 é igual a 4. Se precisar de mais alguma coisa, é só avisar!'},
 {'role': 'user', 'content': 'multiplique por 5'},
 {'role': 'assistant',
  'content': '4 multiplicado por 5 é igual a 20. Se precisar de mais cálculos ou informações, é só me avisar!'}]

# Utilizando o código acima, crie um chatbot customizado para uma empresa. Forneça como contexto no system prompt as principais informações relacionadas a esta empresa:

🏋️‍♂️ Nome da empresa:
Vitality Fitness Center

🏢 Descrição da empresa:
A Vitality Fitness Center é uma academia completa focada em oferecer não apenas musculação de alta qualidade, mas também uma variedade de modalidades para atender todos os perfis de alunos.
Nossa missão é promover saúde, bem-estar e qualidade de vida através de treinos personalizados e acompanhamento de profissionais qualificados.

🎯 Missão:
Transformar vidas por meio do movimento, promovendo saúde física, mental e social.

👓 Visão:
Ser referência em inovação e excelência no segmento fitness e de bem-estar na nossa região até 2030.

💎 Valores:
Comprometimento com resultados

Respeito às individualidades

Inovação constante

Ambiente acolhedor e motivador

Trabalho em equipe

📋 Modalidades oferecidas:
Musculação

Pilates (solo e aparelhos)

Treinamento funcional

Jiu-Jitsu (adulto e infantil)

Cross training

Yoga

Alongamento

Zumba e ritmos

Aulas de HIIT (High-Intensity Interval Training)

👥 Público-alvo:
Jovens e adultos entre 18 e 50 anos

Idosos que buscam melhor qualidade de vida

Crianças e adolescentes para aulas de artes marciais e funcional kids

Pessoas em reabilitação ou buscando treino de baixo impacto (Pilates, Yoga)

🛠️ Estrutura física:
Área de musculação com equipamentos modernos

Estúdios climatizados para aulas de grupo

Espaço de lutas com tatame profissional

Sala de Pilates com equipamentos específicos

Área de descanso e convivência

Vestiários completos

Estacionamento próprio

📍 Localização:
Rua das Palmeiras, 123 – Bairro Saúde Viva – São Paulo, SP

📞 Contato:
Telefone: (11) 99999-1234

WhatsApp: (11) 98888-5678

E-mail: contato@vitalityfitness.com.br

Instagram: @vitalityfitness.br

Site: www.vitalityfitness.com.br

🕑 Horários de funcionamento:
Segunda a sexta: 6h às 22h

Sábado: 8h às 16h

Domingo: fechado

In [ ]:
# configurando chatgpt
import getpass
import os
from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain_core.vectorstores import InMemoryVectorStore
import openai
from openai import OpenAI
from google.colab import userdata

from pathlib import Path
from IPython.display import Audio, display, clear_output, HTML
import time
import pandas as pd
import numpy as np

In [ ]:
# por voz
# STT - speech to text
def transcribe_speech(audio_path):

  audio_file = open(audio_path, "rb")


  transcription = client.audio.transcriptions.create(
      model="gpt-4o-mini-transcribe",
      file=audio_file
  )
  return transcription.text



def resposta_ia(messages):
  #completions api
  completion = client.chat.completions.create(
      model="gpt-4o-mini",
      messages= messages
  )
  return completion.choices[0].message.content



def generate_speech(text):
  speech_file_path = "speech.mp3"

  with client.audio.speech.with_streaming_response.create(
      model="gpt-4o-mini-tts",
      voice="coral",
      input=text,
      instructions="Responda de forma empolgada como se você fosse um mineiro",
  ) as response:
      response.stream_to_file(speech_file_path)

def generate_speech2(text, i=0):
  speech_file_path = f"{i}.mp3"

  with client.audio.speech.with_streaming_response.create(
      model="gpt-4o-mini-tts",
      voice="alloy",
      input=text,
      instructions="Você é um usuário de academia",
  ) as response:
      response.stream_to_file(speech_file_path)

In [ ]:
generate_speech2(text='Oi meu nome é José, gostaria de saber o horário de funcionamento', i=0)
generate_speech2(text='Obrigado. Vocês têm aula de pilates?', i=1)
generate_speech2(text='Obrigado. Até mais tarde', i=2)

In [ ]:
transcribe_speech('/content/0.mp3')

'Oi, meu nome é José. Gostaria de saber o horário de funcionamento.'

In [ ]:
display(Audio(f"0.mp3", autoplay=True))
clear_output(wait=True)
time.sleep(16)

In [ ]:
#voz
msgs= [{
            "role": "system",
            "content": """Você é o Zé, o assistente de atendimento da Academia Vitality Fitness. Responda as dúvidas dos clientes com respostas curtas, de forma animada com base no contexto abaixo:


            🏋️‍♂️ Nome da empresa:
Vitality Fitness Center

🏢 Descrição da empresa:
A Vitality Fitness Center é uma academia completa focada em oferecer não apenas musculação de alta qualidade, mas também uma variedade de modalidades para atender todos os perfis de alunos.
Nossa missão é promover saúde, bem-estar e qualidade de vida através de treinos personalizados e acompanhamento de profissionais qualificados.

🎯 Missão:
Transformar vidas por meio do movimento, promovendo saúde física, mental e social.

👓 Visão:
Ser referência em inovação e excelência no segmento fitness e de bem-estar na nossa região até 2030.

💎 Valores:
Comprometimento com resultados

Respeito às individualidades

Inovação constante

Ambiente acolhedor e motivador

Trabalho em equipe

📋 Modalidades oferecidas:
Musculação

Pilates (solo e aparelhos)

Treinamento funcional

Jiu-Jitsu (adulto e infantil)

Cross training

Yoga

Alongamento

Zumba e ritmos

Aulas de HIIT (High-Intensity Interval Training)

👥 Público-alvo:
Jovens e adultos entre 18 e 50 anos

Idosos que buscam melhor qualidade de vida

Crianças e adolescentes para aulas de artes marciais e funcional kids

Pessoas em reabilitação ou buscando treino de baixo impacto (Pilates, Yoga)

🛠️ Estrutura física:
Área de musculação com equipamentos modernos

Estúdios climatizados para aulas de grupo

Espaço de lutas com tatame profissional

Sala de Pilates com equipamentos específicos

Área de descanso e convivência

Vestiários completos

Estacionamento próprio

📍 Localização:
Rua das Palmeiras, 123 – Bairro Saúde Viva – São Paulo, SP

📞 Contato:
Telefone: (11) 99999-1234

WhatsApp: (11) 98888-5678

E-mail: contato@vitalityfitness.com.br

Instagram: @vitalityfitness.br

Site: www.vitalityfitness.com.br

🕑 Horários de funcionamento:
Segunda a sexta: 6h às 22h

Sábado: 8h às 16h

Domingo: fechado
            """
        }]

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

for i in range(3):
  audio_path =  f"/content/{str(i)}.mp3"

  display(Audio(audio_path, autoplay=True))
  clear_output(wait=True)
  time.sleep(5)

  pergunta = transcribe_speech(audio_path= audio_path)
  print("USUARIO: ", pergunta)
  msgs.append({"role": "user", "content": pergunta})

  resposta = resposta_ia(msgs)
  print("\nALEXA: ", resposta)
  generate_speech(resposta)
  display(Audio(f"speech.mp3", autoplay=True))
  clear_output(wait=True)
  time.sleep(16)
  msgs.append({"role": "assistant", "content": resposta})

USUARIO:  Obrigado. Até mais tarde.

ALEXA:  Até mais tarde, José! 😊💪 Estamos te esperando para treinar e se divertir! Qualquer coisa, é só chamar! 🥳🏋️‍♂️


# Tool Calling

In [ ]:
import openai
from openai import OpenAI
from google.colab import userdata

In [ ]:
# uso simples
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

response = client.responses.create(
    model="gpt-4o-mini",
    input="quem foi o último campeão brasileiro?",
    temperature=0,
    max_output_tokens=500,

)

print(response.output_text)

O último campeão brasileiro, até a minha última atualização, foi o Palmeiras, que conquistou o título do Campeonato Brasileiro de 2022. Para informações mais recentes, recomendo verificar fontes atualizadas, como sites de esportes ou notícias.


In [ ]:
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

In [ ]:
from openai import OpenAI
import json
from google.colab import userdata

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current temperature for provided coordinates in celsius.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number", },
                "longitude": {"type": "number" }
            },
            "required": ["latitude", "longitude"],
            "additionalProperties": False
        },
        "strict": True
    }
}]

messages = [{"role": "user", "content": "tempo em SP"}]

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
)

In [ ]:
completion.choices[0].message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_p9jpX8sKZtcFMI0XfunuskIS', function=Function(arguments='{"latitude":-23.5505,"longitude":-46.6333}', name='get_weather'), type='function')]

In [ ]:
completion.choices[0].message.content

In [ ]:
tool_call = completion.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

result = get_weather(args["latitude"], args["longitude"])
result

20.5

In [ ]:
args

{'latitude': -23.5505, 'longitude': -46.6333}

In [ ]:
messages.append(completion.choices[0].message)  # append model's function call message
messages.append({                               # append result message
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": str(result)
})

completion_2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
)
completion_2.choices[0].message.content


'A temperatura atual em São Paulo é de 20,5°C.'

##Exercício 1: Criando um Agente com Tool Calling para Responder Perguntas com Cálculos

Neste exercício, você irá usar a funcionalidade de tool calling (chamada de ferramentas) da OpenAI para construir um agente que responde perguntas que requerem cálculo matemático, chamando uma função Python definida por você.

Modelos de linguagem são muito bons em linguagem natural, mas nem sempre são precisos com cálculos ou lógica matemática. Ao utilizar o recurso de tool calling, podemos conectá-los a funções externas confiáveis — como uma calculadora Python, sistema de busca ou banco de dados — para obter respostas precisas.

Para te auxiliar, use a função eval(expressao) do python para resolver cálculos matemáticos.


##Exercício 2: Criando um Agente com Tool Calling para Responder Perguntas sobre contagem de caracteres (ex. quantas letras 'o' tem na palavra capivara?)

Neste exercício, você irá usar a funcionalidade de tool calling (chamada de ferramentas) da OpenAI para construir um agente que responde perguntas sobre contagem de caracteres, chamando uma função Python definida por você.

Modelos de linguagem são muito bons em linguagem natural, mas nem sempre são precisos com cálculos ou lógica matemática ou contagem devido a tokenização. Ao utilizar o recurso de tool calling, podemos conectá-los a funções externas confiáveis — como uma calculadora Python, sistema de busca ou banco de dados — para obter respostas precisas.


In [ ]:
'Quanto é 2+3*345'

1037

In [ ]:
@function_tool
def math(expr: str) -> float
  return eval(expr)

In [ ]:
eval("15 * 4")

60